# Elliptic Curves Pseudo Random Number Generators

In [35]:
from math import erfc, sqrt
from numpy import set_printoptions
from scipy.special import gammaincc
set_printoptions(precision=4)

## Para importar tiempo

In [171]:
import time
t_0 = time.time()

### Tu código aquí

t_1 = time.time()
total = t_1 - t_0
print(total)

0.0


# Initial Functions

In [173]:
# Finds the m- ary expansion of n
def getExpansion (n ,m):
    listOfDigits =[]
    while n >= m:
        digit =n%m
        listOfDigits . append ( digit )
        n =(n - digit ) // m
    listOfDigits.append(n)
    return listOfDigits
    
def intToText (n):
    t= getExpansion(n ,256)
    myString =''
    for i in t :
        myString = myString + chr(i)
    return myString

def textToInt (s):
    n =0
    k =0
    for i in s :
        n=n +ord( i) *(256** k)
        k=k +1
    return n

# Output the multiplicative inverse of a modulo p
def multInverse (a , p):
    result = extendedGCD (a , p)
    if result [0]!=1: # Error message if a and p are not relatively prime
        s=" Numbers needs to be relatively prime "
        return s
    inv = result [1]% p
    return inv

#extended euclidean algorithm
# Output [r,s,t] satisfying s*a+t*b=r=gcd(a,b)
def extendedGCD(a , b):
    r0 , r=a ,b
    s0 , s =1 ,0
    t0 , t =0 ,1
    while (r >0):
        tempr , temps , tempt =r ,s ,t
        q= r0 // r
        r ,s , t=r0 - q*r ,s0 -q*s ,t0 - q*t
        r0 , s0 , t0 = tempr , temps , tempt
    return [r0 ,s0 , t0 ]

def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res
    
def fast2Power (a ,n ,m):
    res = 1
    while n > 0:
        if n % 2 == 1: #If the bit is 1 multiply by the corresponding square
            res = ( res * a ) % m
        a =( a * a) % m
        n = n // 2
    return res

In [174]:
def findSquareRoot (N ,p) :
    N0 = N % p
    if fast2Power (N0, (p - 1) // 2 ,p) == 1: #Euler ’s criterion
        if p % 4 == 3:
            x1 = fast2Power (N0, (p + 1) // 4 , p) #see assignment 2 exercise 2 theoretical part
            y1 = p - x1
            return [x1 , y1]
        else :
            for i in range(1 , (( p - 1) // 2) + 1):
                if (i * i) % p == N0:
                    x1 = i
                    y1 = p - i
                    return [x1 , y1]
    return []
    
def generateCurve (E , p):
    if isElliptic (E , p) == False :
        print (" This is not an elliptic curve ")
        return None
    A, B = E
    listOfPoints =["O"]
    for x in range (p):
        a =(x**3 + A*x + B) % p
        if a == 0:
            listOfPoints.append ([x ,0])
        if fast2Power (a, (p - 1) // 2, p) == 1: # Euler ’s criterion there are solutions
            y1, y2 = findSquareRoot(a, p)
            listOfPoints.append([x, y1])
            listOfPoints.append([x, y2])
    return listOfPoints

def isElliptic (E, p):
    A, B = E
    discr = (4*( A **3) +27*( B **2) )%p
    return discr != 0

def pointOnCurve (P ,E ,p) :
    if P == "O":
        return True
    else :
        A, B = E
        x, y = P
        return (y **2) %p ==( x **3+ A* x+B) %p

def addPoints(P,Q,E,N):
    A = E[0]
    B = E[1]
    if P == "O":
        return Q
    elif Q == "O":
        return P
    x1, x2 = P[0], Q[0]
    y1, y2 = P[1], Q[1]
    if x1 == x2 % N and y1 == -y2 % N:
        return "O"
    else:
        if P != Q:
            d1 = extendedGCD(x2 - x1, N)[0]
            if d1 != 1:
                return [-1, d1]
            lmbda = (y2 - y1) * multInverse(x2 - x1, N) % N
        else:
            d2 = extendedGCD(2 * y1, N)[0]
            if d2 != 1:
                return [-1, d2]
            lmbda = (3 * fast2Power(x1, 2, N) + A) * multInverse(2 * y1, N) % N
        x3 = (fast2Power(lmbda, 2, N) - x1 - x2) % N
        y3 = (lmbda * (x1 - x3) - y1) % N
        return [x3, y3]
        
def doubleAndAdd(P, n, E, p):
    res = "O"
    while n > 0:
        if n % 2 == 1:
            res = addPoints(res, P, E, p)
        P = addPoints(P, P, E, p)
        n = n // 2
    return res  

In [175]:
E = [5, 12]
p = 13

In [176]:
L = generateCurve(E, p)

In [177]:
len(L)

8

In [178]:
L

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

In [179]:
G = L[1]
G

[0, 5]

# Linear congruential generator

In [181]:
U_0 = [0, 8]
N = 20
U = [addPoints(doubleAndAdd(G, k, E, p), U_0, E, p) for k in range(N)]

In [182]:
doubleAndAdd(G, 2, E, p)

[10, 3]

In [183]:
U[0:8]

[[0, 8], 'O', [0, 5], [10, 3], [2, 11], [7, 0], [2, 2], [10, 10]]

In [184]:
L

['O', [0, 5], [0, 8], [2, 2], [2, 11], [7, 0], [10, 3], [10, 10]]

# Power generator

In [186]:
U_0 = G
e = 7
N = 20
U = [doubleAndAdd(U_0, e**k, E, p) for k in range(N)]

In [187]:
U[0:8]

[[0, 5], [0, 8], [0, 5], [0, 8], [0, 5], [0, 8], [0, 5], [0, 8]]

## Bitcoin curve

In [189]:
E = [0,7] #represents the elliptic curve y^2 = x^3 + 7
p = 31 #the true value is bigger
G = [11, 25] #a point on the elliptic curve
L = generateCurve(E, p)

In [190]:
def mockLinearGenerator(E, p, G, U): # just for testing purposes
    L = [addPoints(doubleAndAdd(G, i, E, p), U, E, p) for i in range(p)]
    return L

In [191]:
def linearGenerator(E, p, G, U):
    L = []
    for i in range(p):
        P = addPoints(doubleAndAdd(G, i, E, p), U, E, p) # P = iG + U
        if P in L:
            return L
        L.append(P)

In [192]:
U =  L[8] #another point on the curve, representng the seed
linearGenerator(E, p, G, U)

[[5, 15], [4, 28], [1, 15], [20, 28], [7, 28], [0, 21], [25, 15]]

In [193]:
mockLinearGenerator(E, p, G, U)

[[5, 15],
 [4, 28],
 [1, 15],
 [20, 28],
 [7, 28],
 [0, 21],
 [25, 15],
 [5, 15],
 [4, 28],
 [1, 15],
 [20, 28],
 [7, 28],
 [0, 21],
 [25, 15],
 [5, 15],
 [4, 28],
 [1, 15],
 [20, 28],
 [7, 28],
 [0, 21],
 [25, 15],
 [5, 15],
 [4, 28],
 [1, 15],
 [20, 28],
 [7, 28],
 [0, 21],
 [25, 15],
 [5, 15],
 [4, 28],
 [1, 15]]

In [194]:
def pointToBit(G):
    k = 2
    if G == 'O':
        return '00'
    bx, by = bin(G[0])[2:], bin(G[1])[2:]
    if len(bx) == 1:
        bx = '0' + bx
    if len(by) == 1:
        by = '1' + by
    bx, by = bx[-k:], by[-k:]
    return [bx, by]

In [195]:
def generteBitSequence(L):
    s = ''
    for k in L:
        bk = pointToBit(k)
        s += bk[0] + bk[1]
    return s   

In [196]:
L = [[5, 15], [4, 28], [1, 15], [20, 28], [7, 28], [0, 21], [25, 15]]
bL = generteBitSequence(L)
bL

'0111000001110000110000010111'

# Monobit test

In [1]:
def countOnesList(L):
    """
    Count the number of extra ones of a given list of bits.
    
    :param L: a list of 1s and 0s.
    :return: the number of extra 1s compared to 0s (negative values admitted).
            countOnes([0,1,0,1]) = 0 (no extra ones)
            countOnes([0,1,1,1]) = 2 (two extra ones)
            countOnes([0,1,0,0]) = -2 (two extra zeros)
    """
    Ones = [2*i - 1 for i in L]
    return sum(Ones)

def frequencyMonoTest(L, n, sig):
    """
    Evaluates the proportion of 1s with respect to 0s.

    :param L: a list of 1s and 0s
    :param n: the size of such sequence
    :param sig: the level of significance of the test
    :return: a P-value for the hypothesis that s is random
    """
    extraOnes = countOnesList(L)
    stat = abs(extraOnes) / sqrt(n)
    z = stat/sqrt(2)
    pVal = erfc(z)
    hyp = pVal >= sig
    return dict([(pVal, hyp)])

In [57]:
s = '1100100100001111110110101010001000100001011010001100001000110100110001001100011001100010100010111000'
L = [int(k) for k in s]
n = 100
alpha = 0.01
frequencyMonoTest(L,n,alpha)

{0.10959858339911599: True}

# Frequency test M-Blocks

In [207]:
def listBlocks(L, n, N, M):
    """
    generates a list of N blocks of size M from a given list L of n elements
    """
    R = [L[i*M:(i+1)*M] for i in range(N)]
    return R

def frequencyBlockTest(L, n, N, M, sig):
    """
    Tests the distribution of ones of a given bit sequence L of size n within each M-block.

    :param L: a list of ones and zeros,
    :param n: the size of the list,
    :param N the number of the blocks,
    :param M: the size of each block,
    :param sig: level of significance,
    
    :return: accept/reject the randomness of L
    """
    Blocks = listBlocks(L, n, M)
    Ones = [b.count(1)/M for b in Blocks] #containing the proportion of ones within each block
    Terms = [(k - 1/2)**2 for k in Ones]
    chiSquared = 4 * M * sum(Terms)
    gammaTest = gammaincc(N/2, chiSquared/2)
    hyp = gammaTest >= sig
    return dict([(gammaTest, hyp)])



In [337]:
bits = '1100100100001111110110101010001000100001011010001100001000110100110001001100011001100010100010111000'
L = [int(b) for b in bits]
n = 100
N, M = 10, 10
sig = 0.01
frequencyBlockTest(L, n, N, M, sig)

{0.7064384496412808: True}